In [0]:
from dataclasses import dataclass
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from datetime import *
from delta.tables import *

In [0]:
# Create class for Bronze Layer
@dataclass
class BronzeLayer:
    file_path:str
    header:bool
    delimiter:str
    table_name:str

    def __post_init__(self) -> None:
        self.format_type = self.file_path.split('.')[-1]
        self.target_table_bronze = f'{self.table_name}_bronze'

    @classmethod
    def from_config_table(cls, pipeline_name:str) -> "BronzeLayer":
        conf = (spark.table("netflix.config_table")
               .filter(col("pipeline_name") == pipeline_name)
               .select("file_path", "header", "delimiter", "table_name")
               .first())
        return cls(
            file_path = conf.file_path
            , header = conf.header
            , delimiter = conf.delimiter
            , table_name = conf.table_name
        )
    def read_from_file(self) -> DataFrame:
        df = (
            spark.read.format(self.format_type)
            .option("header", self.header)
            .option("delimiter", self.delimiter)
            .load(self.file_path)
        )
        # return with another metadata columns
        return (
            df
            .withColumn("_load_dt", current_date())
            .withColumn("_load_dttm", current_timestamp())
            .withColumn("_file_name", col("_metadata.file_name"))
            .withColumn("_file_path", col("_metadata.file_path"))
            .withColumn("_file_size", col("_metadata.file_size"))
            .withColumn("_file_mod", col("_metadata.file_modification_time"))
        )
    def load_from_bronze_table(self, raw_df: DataFrame) -> None:
        # Ensure the bronze table is initialized with correct settings before loading
        self._init_bronze_table() 

        # write data to bronze table
        raw_df.write.mode("append").saveAsTable(self.target_table_bronze)
        print(f"Table {self.target_table_bronze} loaded")

    def _init_bronze_table(self) -> None:
        # created schema for first time and enable CDF to table
        ## 1. First check if table exists or not
        if spark.catalog.tableExists(self.target_table_bronze):
            print(f"Table {self.target_table_bronze} already exists.")
            return # end method immediately
            
        # 2. If table not exitsts. We create table
        else:
            print(f"Table {self.target_table_bronze} does not exist. Initializing...")
            
            # Retrive schema table (read with out data limit 0)
            source_df = self.read_from_file().limit(0)
            
            # Create table from Schema and enable CDF with Python API
            (source_df.write
             .format("delta")
             .option("delta.enableChangeDataFeed", "true")
             .mode("ignore") # prevent other class creating table before this action
             .saveAsTable(self.target_table_bronze))
            
            print(f"Table {self.target_table_bronze} created successfully with CDF enabled.")

In [0]:
# spark.table("workspace.netflix.config_table").display()

In [0]:
# %sql
# truncate table workspace.netflix.netflix_bronze_bronze

In [0]:
# b = BronzeLayer.from_config_table("netflix")
# bronze_df = b.read_from_file()
# b.load_to_bronze_table(bronze_df)

# spark.table("workspace.netflix.netflix_bronze_bronze").display()

In [0]:
# %sql
# describe extended  workspace.netflix.netflix_bronze_bronze

In [0]:
# spark.table("workspace.netflix.netflix_bronze_bronze").display()

In [0]:
# %sql
# DESCRIBE HISTORY workspace.netflix.netflix_bronze_bronze

ต้องทำการสำรวจข้อมูลก่อน(ทำแล้ว) โดยเราต้องมีการทำความสะอาดข้อมูล ทำ Normalization Standardization ประมาณนี้
📋 ลำดับการ Clean Data ที่จะใช้ในโปรเจคนี้
    0. We need to create control table(for first time) to store last version of bronze table that loaded to silver table already # successes
    1. Load data from Bronze by use feature like CDF for get only new data(by compare current version and last version of data which processed already in control table) and Add serogete key to table # successes
    2. Trim String Column & Change Data Type
        1.1 Standardize Date & Business 
    3. Get Invalid Record (Data Quality Audit)
    4. Get Key Duplicate (Deduplication Logic)
    5. Get Row Duplicate
    6. Get only bad record and save to bad record table
    7. After get bad record left anti join for get final record and save it into Silver layer.


In [0]:
# create class for silver layer
@dataclass
class SilverLayer:
    table_name: str
    schema_detail: dict[str, str]
    keys: list[str]
    write_mode: str

    def __post_init__(self) -> None:
        self.bronze_table_name = f"{self.table_name}_bronze"
        self.silver_table_name = f"{self.table_name}_silver"
        self.bad_record_table_name = f"{self.table_name}_bad_record"
        self.data_col = [col_name for col_name in self.schema_detail.keys()]
        self.invalid_rule = {"int": "^[0-9]+$", "date": "^\\d{4}-\\d{2}-\\d{2}$"}

    @classmethod
    def from_config_table(cls, pipeline_name: str) -> "SilverLayer":
        conf = (
            spark.table("workspace.netflix.config_table")
            .filter(col("pipeline_name") == pipeline_name)
            .select(
                "table_name", "schema_detail", "keys", "write_mode"
            )
            .first()
        )
        return cls(
            table_name=conf.table_name,
            schema_detail=conf.schema_detail,
            keys=conf.keys,
            write_mode=conf.write_mode,
        )
    
    def _get_earliest_readable_version(self, delta_table: DeltaTable) -> int:
        """
        Determine the earliest version we can safely read from.
        Handles TRUNCATE operations and retention policies.
        
        Returns:
        - Version after last TRUNCATE (if any)
        - Or version 0 if no TRUNCATE exists
        """
        history = delta_table.history()
        
        # Find last TRUNCATE operation
        last_truncate = (
            history.filter(col("operation") == "TRUNCATE")
            .orderBy(col("version").desc())
            .first()
        )
        
        if last_truncate:
            # Start from version after TRUNCATE
            earliest_version = last_truncate["version"] + 1
            print(f"Detected TRUNCATE at version {last_truncate['version']}, starting from version {earliest_version}")
            return earliest_version
        
        # No TRUNCATE - start from version 0
        return 0
    
    def _get_safe_starting_version(self, delta_table: DeltaTable, last_processed_version: int) -> int:
        """
        Check if a TRUNCATE occurred after last_processed_version.
        If yes, return version after TRUNCATE. Otherwise, return last_processed + 1.
        
        This handles the case where TRUNCATE happens during incremental runs (not just first run).
        """
        history = delta_table.history()
        
        # Find TRUNCATE operations after last_processed_version
        truncates_after_last = (
            history.filter(
                (col("operation") == "TRUNCATE") & 
                (col("version") > last_processed_version)
            )
            .orderBy(col("version").desc())
            .first()
        )
        
        if truncates_after_last:
            truncate_version = truncates_after_last["version"]
            safe_version = truncate_version + 1
            print(f"⚠️  TRUNCATE detected at version {truncate_version} (after last processed: {last_processed_version})")
            print(f"   Adjusting starting version from {last_processed_version + 1} to {safe_version}")
            return safe_version
        
        # No TRUNCATE in range - safe to continue from last_processed + 1
        return last_processed_version + 1
    
    def read_from_bronze_table_add_sk(self) -> DataFrame:
        """
        Read new data from bronze table using Change Data Feed.
        Returns DataFrame with data columns + _sk surrogate key.
        
        Logic:
        - First run: Read from earliest readable version to latest
        - Subsequent runs: Check for TRUNCATE in range, adjust starting version if needed
        - No new data: Return empty DataFrame with schema, update check timestamp
        - Handles TRUNCATE operations and retention policies
        """
        # Initialize control table
        control_table = ControlTable("workspace.netflix.control_table")
        control_table.init_control_table()
        
        # Get last processed version
        last_state = control_table.get_last_version_for_table(self.bronze_table_name)
        
        # Get Delta table and latest version
        delta_table = DeltaTable.forName(spark, self.bronze_table_name)
        latest_version = delta_table.history(1).select("version").first()["version"]
        
        # Determine starting version
        if last_state is None:
            # First run - find earliest readable version (handles TRUNCATE)
            starting_version = self._get_earliest_readable_version(delta_table)
            print(f"First run: reading CDF from version {starting_version} to {latest_version}")
        else:
            last_processed = last_state["version"]
            
            # Check if there's new data
            if latest_version <= last_processed:
                print(f"No new data: latest version {latest_version} already processed (last: {last_processed})")
                control_table.update_check_timestamp(self.bronze_table_name)
                
                # Return empty DataFrame with correct schema
                return spark.sql(
                    f"SELECT {', '.join(self.data_col)}, CAST(NULL AS LONG) AS _sk "
                    f"FROM {self.bronze_table_name} WHERE 1=0"
                )
            
            # New data available - check for TRUNCATE in the range
            starting_version = self._get_safe_starting_version(delta_table, last_processed)
            print(f"Reading CDF from version {starting_version} to {latest_version}")
        
        # Handle edge case: starting version > latest version
        if starting_version > latest_version:
            print(f"No data in range: starting {starting_version} > latest {latest_version}")
            control_table.update_check_timestamp(self.bronze_table_name)
            return spark.sql(
                f"SELECT {', '.join(self.data_col)}, CAST(NULL AS LONG) AS _sk "
                f"FROM {self.bronze_table_name} WHERE 1=0"
            )
        
        # Read Change Data Feed
        cdf_df = (
            spark.read.format("delta")
            .option("readChangeFeed", "true")
            .option("startingVersion", starting_version)
            .option("endingVersion", latest_version)
            .table(self.bronze_table_name)
        )
        
        # Update control table with new processed version
        control_table.update_last_version_for_table(
            self.bronze_table_name, 
            latest_version, 
            datetime.now()
        )
        
        # Select data columns and add surrogate key
        return (
            cdf_df
            .select(*self.data_col)
            .withColumn("_sk", monotonically_increasing_id())
        )

    def trim_data(self, df: DataFrame) -> DataFrame:
        """
        Trim all string columns to remove leading/trailing whitespace.
        Uses select() to avoid performance issues from .withColumn() in a loop.
        """
        # Pre-compute schema to avoid repeated Analyze RPCs
        df_columns = df.columns
        
        trim_exprs = [
            trim(col(col_name)).alias(col_name) if col_type == "string" else col(col_name)
            for col_name, col_type in self.schema_detail.items()
        ]
        # Include _sk if it exists
        if "_sk" in df_columns:
            trim_exprs.append(col("_sk"))
        
        return df.select(*trim_exprs)
    
    def change_data_type(self, df: DataFrame) -> DataFrame:
        pass # I will create for next day 02/06/2026

In [0]:
# Create class for control table
@dataclass
class ControlTable:
    control_table_name: str

    def init_control_table(self) -> None:
        if spark.catalog.tableExists(self.control_table_name):
            print(f"Table {self.control_table_name} already exists.")
            return
        
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {self.control_table_name} (
                table_name STRING NOT NULL,
                last_processed_version LONG NOT NULL,
                last_processed_timestamp TIMESTAMP NOT NULL,
                last_check_timestamp TIMESTAMP NOT NULL
            ) 
            USING DELTA
            TBLPROPERTIES (
                delta.autoOptimize.optimizeWrite = true,
                delta.autoOptimize.autoCompact = true
            )
        """)
        print(f"Table {self.control_table_name} created.")

    def get_last_version_for_table(self, tracked_table_name: str) -> dict:
        """
        Returns dict with 'version' and 'timestamp', or None if table not tracked yet.
        """
        df = spark.table(self.control_table_name).filter(col("table_name") == tracked_table_name)
        row = df.first()
        
        if row:
            return {
                "version": row.last_processed_version,
                "timestamp": row.last_processed_timestamp
            }
        return None

    def update_last_version_for_table(self, tracked_table_name: str, version: int, timestamp) -> None:
        """
        UPSERT (merge) logic: updates existing row or inserts new row.
        Also updates last_check_timestamp to track when the check was performed.
        """
        # Create a temp view with the new data
        new_data = [(tracked_table_name, version, timestamp, timestamp)]
        columns = ["table_name", "last_processed_version", "last_processed_timestamp", "last_check_timestamp"]
        updates_df = spark.createDataFrame(new_data, columns)
        updates_df.createOrReplaceTempView("control_updates")
        
        # Use MERGE to upsert
        spark.sql(f"""
            MERGE INTO {self.control_table_name} AS target
            USING control_updates AS source
            ON target.table_name = source.table_name
            WHEN MATCHED THEN
                UPDATE SET
                    target.last_processed_version = source.last_processed_version,
                    target.last_processed_timestamp = source.last_processed_timestamp,
                    target.last_check_timestamp = source.last_check_timestamp
            WHEN NOT MATCHED THEN
                INSERT (table_name, last_processed_version, last_processed_timestamp, last_check_timestamp)
                VALUES (source.table_name, source.last_processed_version, source.last_processed_timestamp, source.last_check_timestamp)
        """)
        print(f"Control table updated: {tracked_table_name} -> version {version}")
    
    def update_check_timestamp(self, tracked_table_name: str) -> None:
        """
        Update only the last_check_timestamp to indicate when we last checked for new data.
        Useful when there's no new data but we want to track that we performed a check.
        """
        spark.sql(f"""
            UPDATE {self.control_table_name}
            SET last_check_timestamp = current_timestamp()
            WHERE table_name = '{tracked_table_name}'
        """)
        print(f"Check timestamp updated for {tracked_table_name}")

In [0]:
# ct = ControlTable("workspace.netflix.control_table")
# ct.init_control_table()
# last_num = ct.get_last_version()
# ct.update_last_version(version=5, timestamp=datetime.now())
# print(last_num)

In [0]:
spark.table("workspace.netflix.control_table").display()

In [0]:
spark.table("workspace.netflix.config_table").display()

In [0]:
test = SilverLayer.from_config_table("netflix")
bronze_cdf = test.read_from_bronze_table_add_sk()
trim_df = test.trim_data(bronze_cdf)

In [0]:
trim_df.display()